# 🏛️ Indian Legal Reasoning LLM — Complete Pipeline

**Single notebook combining all 5 stages of the pipeline.**

| Stage | Section | Runtime | GPU |
|-------|---------|---------|-----|
| 1 | Setup & Data Exploration | ~15 min | ✅ |
| 2 | CoT Generation (DeepSeek-R1) | ~1 hr | ❌ |
| 3 | SFT Fine-tuning (LoRA) | ~3 hrs | ✅ |
| 4 | GRPO RL Fine-tuning | ~2 hrs | ✅ |
| 5 | Evaluation & Reporting | ~1 hr | ✅ |

**Checkpoint system:** Each stage saves its outputs to disk. If the notebook crashes,
you can skip completed stages by loading from checkpoints.

---

# ══════════════════════════════════════════════
# STAGE 1 — Environment Setup & Data Exploration
# ══════════════════════════════════════════════

## 1.1 Install Dependencies

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q sentencepiece protobuf huggingface_hub
!pip install -q rouge-score bert-score nltk evaluate
!pip install -q openai pandas numpy tqdm

## 1.2 Imports & Path Setup

In [ ]:
import sys, os, json, re, gc
import torch
import numpy as np
import pandas as pd
from pathlib import Path

# Add project root to path for src/ imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Create output directories
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1.3 Authenticate

In [ ]:
from huggingface_hub import login
from src.config import HF_TOKEN, DEEPSEEK_API_KEY

login(HF_TOKEN)
print(f"HuggingFace: ✅")
print(f"DeepSeek API: {'✅ Found' if DEEPSEEK_API_KEY else '❌ Missing (needed in Stage 2)'}")

## 1.4 Load Base Model

In [ ]:
from src.model_utils import load_base_model

model, tokenizer = load_base_model()

print(f"\nVocab size: {tokenizer.vocab_size}")
print(f"Model type: {model.config.model_type}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num layers: {model.config.num_hidden_layers}")

## 1.5 Sanity Check — Base Model Before Fine-tuning

In [ ]:
from src.model_utils import generate_judgment

sample_facts = """The accused was charged under Section 302 IPC for
allegedly murdering his neighbor during a property dispute.
The prosecution presented three eyewitnesses and forensic evidence.
The defense argued it was a case of self-defense under Section 96 IPC."""

print("=== BASE MODEL OUTPUT (before any fine-tuning) ===")
base_output = generate_judgment(model, tokenizer, sample_facts)
print(base_output)

## 1.6 Load & Explore Datasets

In [ ]:
from src.data_utils import load_legal_datasets, prepare_sft_dataset

ildc, nyaya = load_legal_datasets()

print("\n=== ILDC Dataset ===")
print(f"Columns: {ildc.column_names}")
print(f"Sample (first 300 chars):")
for key, val in ildc[0].items():
    print(f"  {key}: {str(val)[:300]}")

print("\n=== NyayaAnumana Dataset ===")
print(f"Columns: {nyaya.column_names}")
print(f"Sample (first 300 chars):")
for key, val in nyaya[0].items():
    print(f"  {key}: {str(val)[:300]}")

## 1.7 Format for Instruction Tuning

In [ ]:
sft_dataset = prepare_sft_dataset(ildc)

print("\n=== Formatted Sample Preview (first 500 chars) ===")
print(sft_dataset['train'][0]['text'][:500])

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 1 CHECKPOINT                   ║
# ║  Model loaded, datasets explored.        ║
# ║  Free model memory for Stage 2 (API).    ║
# ╚══════════════════════════════════════════╝

# Free GPU memory — Stage 2 doesn't need the model
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Stage 1 complete. GPU memory freed for Stage 2.")

---
# ══════════════════════════════════════════════
# STAGE 2 — CoT Generation via DeepSeek-R1
# ══════════════════════════════════════════════

> **Cost:** ~$1-2 for 500 cases | **GPU:** Not needed | **Runtime:** ~1 hr

In [ ]:
from src.config import COT_DATA_PATH, COT_NUM_SAMPLES

# Check if CoT data already exists (resume from checkpoint)
COT_EXISTS = os.path.exists(COT_DATA_PATH)

if COT_EXISTS:
    with open(COT_DATA_PATH, 'r', encoding='utf-8') as f:
        existing_lines = len(f.readlines())
    print(f"⏩ CHECKPOINT FOUND: {COT_DATA_PATH} has {existing_lines} samples.")
    print(f"   Skipping CoT generation. Delete the file to regenerate.")
else:
    print(f"No checkpoint found. Will generate {COT_NUM_SAMPLES} CoT samples.")

## 2.1 Test Single CoT Generation

In [ ]:
if not COT_EXISTS:
    from src.cot_generator import create_deepseek_client, get_cot_from_teacher
    
    client = create_deepseek_client()
    
    # Test with first case
    test_facts = ildc[0].get('facts', ildc[0].get('text', ''))[:1000]
    result = get_cot_from_teacher(client, test_facts)
    
    print("=== REASONING TRACE (first 500 chars) ===")
    print(result['reasoning'][:500])
    print("\n=== ANSWER (first 500 chars) ===")
    print(result['answer'][:500])
else:
    print("⏩ Skipped — using existing checkpoint.")

## 2.2 Generate Full Batch (500 Cases)

In [ ]:
if not COT_EXISTS:
    from src.cot_generator import generate_cot_batch
    
    cot_data = generate_cot_batch(
        ildc_dataset=ildc,
        client=client,
        n_samples=COT_NUM_SAMPLES,
        save_path=COT_DATA_PATH,
        save_every=50,
    )
    print(f"\n🎉 Generated {len(cot_data)} CoT samples!")
else:
    print("⏩ Skipped — using existing checkpoint.")

## 2.3 Load & Verify CoT Data

In [ ]:
from src.data_utils import load_cot_dataset

cot_dataset = load_cot_dataset(COT_DATA_PATH)

# Inspect a sample
print("=== Formatted CoT Sample (first 500 chars) ===")
print(cot_dataset[0]['text'][:500])

# Stats
lengths = [len(s['text']) for s in cot_dataset]
print(f"\n=== CoT Dataset Stats ===")
print(f"  Total samples: {len(cot_dataset)}")
print(f"  Avg length:    {np.mean(lengths):.0f} chars")
print(f"  Min length:    {np.min(lengths)} chars")
print(f"  Max length:    {np.max(lengths)} chars")

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 2 CHECKPOINT                   ║
# ║  cot_data.jsonl saved to disk.           ║
# ║  Safe to restart from here.              ║
# ╚══════════════════════════════════════════╝
print(f"✅ Stage 2 complete. CoT data saved at: {COT_DATA_PATH}")

---
# ══════════════════════════════════════════════
# STAGE 3 — SFT Fine-tuning with LoRA
# ══════════════════════════════════════════════

> **Runtime:** ~3 hrs on Kaggle T4 | **GPU:** Required (16GB)

In [ ]:
from src.config import SFT_OUTPUT_DIR, SFT_LORA_SAVE_DIR

# Check if SFT model already exists (resume from checkpoint)
SFT_EXISTS = os.path.exists(SFT_LORA_SAVE_DIR) and os.path.exists(
    os.path.join(SFT_LORA_SAVE_DIR, "adapter_config.json")
)

if SFT_EXISTS:
    print(f"⏩ CHECKPOINT FOUND: {SFT_LORA_SAVE_DIR}")
    print(f"   Skipping SFT training. Delete the folder to retrain.")
else:
    print(f"No SFT checkpoint found. Will train from scratch.")

## 3.1 Load Model & Apply LoRA

In [ ]:
if not SFT_EXISTS:
    from src.model_utils import load_base_model, apply_lora
    
    model, tokenizer = load_base_model()
    model, lora_config = apply_lora(model)
    # Expected: trainable params ~12M / 1.5B total (~0.8%)
else:
    print("⏩ Skipped — using existing SFT checkpoint.")

## 3.2 Load CoT Training Data

In [ ]:
if not SFT_EXISTS:
    # Reload CoT dataset if not already in memory
    if 'cot_dataset' not in dir():
        from src.data_utils import load_cot_dataset
        cot_dataset = load_cot_dataset(COT_DATA_PATH)
    
    print(f"Training samples: {len(cot_dataset)}")
else:
    print("⏩ Skipped — using existing SFT checkpoint.")

## 3.3 Train with SFTTrainer

In [ ]:
if not SFT_EXISTS:
    from trl import SFTTrainer, SFTConfig
    from src.config import (
        SFT_EPOCHS, SFT_BATCH_SIZE, SFT_GRAD_ACCUM_STEPS,
        SFT_LEARNING_RATE, SFT_WARMUP_RATIO, SFT_MAX_SEQ_LENGTH,
    )
    
    training_args = SFTConfig(
        output_dir=SFT_OUTPUT_DIR,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM_STEPS,
        learning_rate=SFT_LEARNING_RATE,
        warmup_ratio=SFT_WARMUP_RATIO,
        lr_scheduler_type="cosine",
        bf16=False,
        fp16=True,
        logging_steps=10,
        save_steps=100,
        max_seq_length=SFT_MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=cot_dataset,
        args=training_args,
        peft_config=lora_config,
    )
    
    print("🚀 Starting SFT training...")
    trainer.train()
    
    # Save
    trainer.save_model(SFT_LORA_SAVE_DIR)
    print(f"\n✅ SFT training complete. Saved to: {SFT_LORA_SAVE_DIR}")
else:
    print("⏩ Skipped — using existing SFT checkpoint.")

## 3.4 Quick Validation

In [ ]:
if not SFT_EXISTS:
    from src.model_utils import generate_judgment
    
    sample_facts = """The accused was charged under Section 302 IPC for
    allegedly murdering his neighbor during a property dispute.
    The prosecution presented three eyewitnesses and forensic evidence."""
    
    print("=== SFT MODEL OUTPUT ===")
    print(generate_judgment(model, tokenizer, sample_facts))

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 3 CHECKPOINT                   ║
# ║  LoRA adapter saved to disk.             ║
# ║  Free memory for Stage 4.                ║
# ╚══════════════════════════════════════════╝

if not SFT_EXISTS:
    del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"✅ Stage 3 complete. LoRA checkpoint at: {SFT_LORA_SAVE_DIR}")

---
# ══════════════════════════════════════════════
# STAGE 4 — GRPO Reinforcement Learning
# ══════════════════════════════════════════════

> **Runtime:** ~2 hrs on Kaggle T4 | **GPU:** Required (16GB)

In [ ]:
from src.config import GRPO_OUTPUT_DIR, GRPO_FINAL_SAVE_DIR

# Check if GRPO model already exists (resume from checkpoint)
GRPO_EXISTS = os.path.exists(GRPO_FINAL_SAVE_DIR) and os.path.exists(
    os.path.join(GRPO_FINAL_SAVE_DIR, "adapter_config.json")
)

if GRPO_EXISTS:
    print(f"⏩ CHECKPOINT FOUND: {GRPO_FINAL_SAVE_DIR}")
    print(f"   Skipping GRPO training. Delete the folder to retrain.")
else:
    print(f"No GRPO checkpoint found. Will train from SFT model.")

## 4.1 Load SFT Model

In [ ]:
if not GRPO_EXISTS:
    from src.model_utils import load_finetuned_model
    
    model, tokenizer = load_finetuned_model(SFT_LORA_SAVE_DIR)
    print("✅ SFT model loaded for GRPO training")
else:
    print("⏩ Skipped — using existing GRPO checkpoint.")

## 4.2 Test Reward Functions

In [ ]:
from src.reward_functions import (
    reward_has_legal_citation,
    reward_has_reasoning,
    reward_bilingual,
    ALL_REWARD_FUNCS,
)

# Test with sample completions
test_completions = [
    "Under Section 302 IPC, the accused is guilty. <think>Therefore, applying the ratio decidendi...</think> अतः दोषी है।",
    "The person did something bad.",
]

print("=== Reward Function Tests ===")
print(f"  Legal citation: {reward_has_legal_citation(test_completions)}")
print(f"  Reasoning:      {reward_has_reasoning(test_completions)}")
print(f"  Bilingual:      {reward_bilingual(test_completions)}")
print(f"\n  Good sample total: {sum(r[0] for r in [reward_has_legal_citation(test_completions), reward_has_reasoning(test_completions), reward_bilingual(test_completions)]):.1f}")
print(f"  Bad sample total:  {sum(r[1] for r in [reward_has_legal_citation(test_completions), reward_has_reasoning(test_completions), reward_bilingual(test_completions)]):.1f}")

## 4.3 Run GRPO Training

In [ ]:
if not GRPO_EXISTS:
    from trl import GRPOTrainer, GRPOConfig
    from src.config import (
        GRPO_LEARNING_RATE, GRPO_BATCH_SIZE, GRPO_GRAD_ACCUM_STEPS,
        GRPO_NUM_GENERATIONS, GRPO_MAX_NEW_TOKENS, GRPO_EPOCHS,
    )
    from src.data_utils import load_cot_dataset
    
    # Reload CoT dataset if needed
    if 'cot_dataset' not in dir():
        cot_dataset = load_cot_dataset(COT_DATA_PATH)
    
    grpo_config = GRPOConfig(
        output_dir=GRPO_OUTPUT_DIR,
        learning_rate=GRPO_LEARNING_RATE,
        per_device_train_batch_size=GRPO_BATCH_SIZE,
        gradient_accumulation_steps=GRPO_GRAD_ACCUM_STEPS,
        num_generations=GRPO_NUM_GENERATIONS,
        max_new_tokens=GRPO_MAX_NEW_TOKENS,
        num_train_epochs=GRPO_EPOCHS,
        fp16=True,
        report_to="none",
    )
    
    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=ALL_REWARD_FUNCS,
        args=grpo_config,
        train_dataset=cot_dataset,
    )
    
    print("🚀 Starting GRPO training...")
    grpo_trainer.train()
    
    grpo_trainer.save_model(GRPO_FINAL_SAVE_DIR)
    print(f"\n✅ GRPO training complete. Saved to: {GRPO_FINAL_SAVE_DIR}")
else:
    print("⏩ Skipped — using existing GRPO checkpoint.")

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  ✅ STAGE 4 CHECKPOINT                   ║
# ║  Final GRPO model saved to disk.         ║
# ║  Free memory for Stage 5 (eval).         ║
# ╚══════════════════════════════════════════╝

if not GRPO_EXISTS and 'grpo_trainer' in dir():
    del model, grpo_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"✅ Stage 4 complete. Final model at: {GRPO_FINAL_SAVE_DIR}")

---
# ══════════════════════════════════════════════
# STAGE 5 — Evaluation & Reporting
# ══════════════════════════════════════════════

> **Runtime:** ~1 hr | **GPU:** Required | **Cost:** ~$0.05 (DeepSeek Chat)

## 5.1 Load Both Models for Comparison

In [ ]:
from src.model_utils import load_base_model, load_finetuned_model
from src.config import EVAL_N_SAMPLES, JUDGE_N_SAMPLES, EVAL_REPORT_PATH, ILDC_DATASET
from datasets import load_dataset

# Load base model
base_model, tokenizer = load_base_model()
print()

# Load fine-tuned model (use GRPO if available, else SFT)
ft_path = GRPO_FINAL_SAVE_DIR if os.path.exists(GRPO_FINAL_SAVE_DIR) else SFT_LORA_SAVE_DIR
finetuned_model, _ = load_finetuned_model(ft_path)
print(f"\nUsing fine-tuned model from: {ft_path}")

# Load test set
ildc_test = load_dataset(ILDC_DATASET, split="test")
print(f"Test cases: {len(ildc_test)}")

## 5.2 Quick Sanity Check

In [ ]:
from src.model_utils import generate_judgment

sample = ildc_test[0]
print("=== FACTS (first 400 chars) ===")
print(sample['text'][:400])
print("\n=== FINE-TUNED MODEL OUTPUT ===")
print(generate_judgment(finetuned_model, tokenizer, sample['text']))

## 5.3 ROUGE + BERTScore (100 samples)

In [ ]:
from src.evaluation import evaluate_rouge_bert, print_comparison_table

print("📊 Evaluating BASE model...")
base_scores = evaluate_rouge_bert(base_model, tokenizer, ildc_test, n_samples=EVAL_N_SAMPLES)

print("\n📊 Evaluating FINE-TUNED model...")
ft_scores = evaluate_rouge_bert(finetuned_model, tokenizer, ildc_test, n_samples=EVAL_N_SAMPLES)

print_comparison_table(base_scores, ft_scores)

## 5.4 LLM-as-Judge (30 samples)

In [ ]:
from src.evaluation import run_llm_judge_eval, print_judge_scores

print("⚖️ Running LLM Judge on fine-tuned model...")
judge_results = run_llm_judge_eval(
    finetuned_model, tokenizer, ildc_test,
    n_samples=JUDGE_N_SAMPLES,
)

print_judge_scores(judge_results)

## 5.5 Side-by-Side Comparison

In [ ]:
from src.evaluation import print_side_by_side

for idx in [0, 5, 12, 20]:
    print_side_by_side(base_model, finetuned_model, tokenizer, ildc_test, idx)
    print()

## 5.6 Export Final Report

In [ ]:
from src.evaluation import export_report

df = export_report(judge_results, save_path=EVAL_REPORT_PATH)

if 'overall' in df.columns:
    print(f"\n🏆 Top 5 best cases:")
    print(df.nlargest(5, 'overall')[['case_id', 'overall', 'feedback']])
    print(f"\n⚠️  Bottom 5 cases:")
    print(df.nsmallest(5, 'overall')[['case_id', 'overall', 'feedback']])

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  🎉 PIPELINE COMPLETE                    ║
# ║                                          ║
# ║  Outputs saved:                          ║
# ║  • cot_data.jsonl (CoT training data)    ║
# ║  • indian-legal-1.5b-lora/ (SFT model)   ║
# ║  • indian-legal-1.5b-final/ (GRPO model) ║
# ║  • legal_eval_report.csv (eval results)  ║
# ╚══════════════════════════════════════════╝

print("\n" + "=" * 60)
print("  🎉 INDIAN LEGAL LLM PIPELINE — COMPLETE")
print("=" * 60)
print(f"\n  📁 Saved artifacts:")
for path in [COT_DATA_PATH, SFT_LORA_SAVE_DIR, GRPO_FINAL_SAVE_DIR, EVAL_REPORT_PATH]:
    exists = '✅' if os.path.exists(path) else '❌'
    print(f"     {exists} {path}")
print()